# Creates a mosaic.json from a STAC-Geoparquet

Will read and write to/from S3.

In [41]:
# Read and write locally for now.
from os import makedirs
from pystac import ItemCollection
from cogeo_mosaic.mosaic import MosaicJSON
from cogeo_mosaic.backends import MosaicBackend
from rustac import search_sync
from shapely.geometry import shape, mapping

# input_stac_geoparquet = "https://s3.us-west-2.amazonaws.com/data.ldn.auspatious.com/ausp_ls_geomad/0-0-2/ausp_ls_geomad.parquet"
# output_mosaic_json_path = "https://s3.us-west-2.amazonaws.com/data.ldn.auspatious.com/mosaics/"
input_stac_geoparquet = "/Users/wj/Downloads/ausp_ls_geomad.parquet"
output_mosaic_json_path = "/Users/wj/Downloads/mosaics/"

makedirs(output_mosaic_json_path, exist_ok=True)

# TODO: Run for each year.
year = "2020"

item_collection = search_sync(input_stac_geoparquet, datetime=year)
items = ItemCollection(item_collection)
features = [f.to_dict() for f in items]

# cogeo-mosaic's _create_mosaic assumes Polygon geometries.
# Use shapely to convert any MultiPolygon to its convex hull Polygon.
print(len(features), "features read from STAC GeoParquet.")
for feat in features:
    geom = shape(feat["geometry"])
    # TODO: This is a hack. Needed because cogeo_mosaic expects Polygon geoms. We could change this to use the bbox or a convex hull.
    geom = geom.convex_hull  # Convert to convex hull to ensure it's a Polygon. Many features are Multipolygons.
    feat["geometry"] = mapping(geom)

# The default accessor looks for feature["properties"]["path"], which
# STAC items don't have.  Use the self link href as the asset identifier.
def stac_accessor(feature):
    # TODO: Just using red as a placeholder. For the prediction data it will be single band classification.
    return feature["assets"]["red"]["href"]

mosaic = MosaicJSON.from_features(
    features,
    minzoom=5,
    maxzoom=14,
    accessor=stac_accessor,
)

print(f"quadkey_zoom: {mosaic.quadkey_zoom}, tiles: {len(mosaic.tiles)}")

with MosaicBackend(f"{output_mosaic_json_path}/mosaic_{year}.json", mosaic_def=mosaic) as m:
        m.write(overwrite=True)
        print("Mosaic JSON written.")


# Check titiler_lambda.md for visualising this mosaic.json.

7 features read from STAC GeoParquet.
quadkey_zoom: 5, tiles: 36
Mosaic JSON written.
